In [0]:
!pip install pyspark==3.5.1
!pip install apache-sedona==1.6.1

In [0]:
%scala
import org.apache.sedona.spark.SedonaContext

// Initialize Sedona so ST_* functions are registered
val sedona = SedonaContext.create(spark)

val catalog       = "apa_prod"
val sourceTable   = s"$catalog.bfp"
val targetTable   = s"$catalog.osm_bfp"

// Read source, project + convert in one step
val osmBfpDf = spark.read.table(sourceTable)
  .selectExpr(
    "bfpId              AS bfp_id",
    "ST_GeomFromWKT(wkt) AS geometry",
    "licenseZone        AS country_iso"
  )

display(osmBfpDf)

In [0]:
%scala
// Write as a Delta table (overwrite; use "append" if you're adding to it)
osmBfpDf.write
  .format("delta")
  .mode("overwrite")
  .option("overwriteSchema", "true")
  .saveAsTable(targetTable)